# DMC Governed Inference Architecture (v1.1)
**Current Branch:** `feature/v1.1-governed-inference`

This notebook implements Unsloth 4-bit Quantization, LoRA Adapters for Domain Scoping, and Citation-Anchored Prompt Engineering.

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# ==========================================
# SECTION 1: ENTERPRISE EDGE CONFIGURATION (UNSLOTH)
# ==========================================

print("[System] Configuring 4-bit Quantization for Edge Inference...")

# 1. Configuration for Memory Efficiency (Matches 'Meta Wisdom' Edge Req)
max_seq_length = 2048 
dtype = None # Auto detection (Float16/Bfloat16)
load_in_4bit = True # 4-bit quantization for offline Android deployment

# 2. Load Base Model with Governance Token
# We use the Instruct version to enforce refusal taxonomies
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# ==========================================
# SECTION 2: LoRA ADAPTER INJECTION (DOMAIN SCOPING)
# ==========================================

print("[System] Injecting Low-Rank Adapters (LoRA) for Domain Scoping...")

# This configuration restricts the model's learning to specific parameters,
# ensuring it adapts to the 'Knowledge Graph' without catastrophic forgetting.
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Optimized Rank
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj", # Attention mechanisms
        "gate_proj", "up_proj", "down_proj",   # Feed-forward networks
    ],
    lora_alpha = 16,
    lora_dropout = 0, # Set to 0 for deterministic enterprise use
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None, 
)

# ==========================================
# SECTION 3: CITATION-ANCHORED PROMPT ENGINEERING
# ==========================================

# This template implements the "Strict Citation" requirement.
# It forces the model to act as a Retrieval Agent, not a Creative Writer.

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are a domain-specific assistant. You must answer the user's question using ONLY the provided Context. 
Every claim must be followed by a citation ID in brackets (e.g., [Doc-1]).
If the answer is not in the context, state "I cannot find this information in the approved sources."

### Input:
{input}

### Context:
{context}

### Response:
{}"""

EO_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN generation

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    contexts     = examples["context"] # Added Context field for RAG
    outputs      = examples["output"]
    texts = []
    for instruction, input, context, output in zip(instructions, inputs, contexts, outputs):
        # Format with the strict governance template
        text = alpaca_prompt.format(input=input, context=context) + output + EO_TOKEN
        texts.append(text)
    return { "text" : texts, }

# ==========================================
# SECTION 4: INFERENCE SIMULATION (CITATION CHECK)
# ==========================================

def run_governed_inference(question, retrieval_context):
    """
    Simulates the on-device inference pass with citation enforcement.
    """
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference
    
    # Construct the governed prompt
    prompt = alpaca_prompt.format(
        input=question,
        context=retrieval_context
    )
    
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 128, 
        use_cache = True,
        temperature = 0.3 # Low temp for factual accuracy
    )
    
    return tokenizer.batch_decode(outputs)

# Example usage print
print("\n[System] Model Configured. Ready for Governed Inference.")
# print(run_governed_inference("What is faith?", "Faith is... [Doc-1]"))